**05 · Descriptores invariantes, recuperación y ModelNet**

Ejecutar desde el repositorio point-field con `python -m pip install -e ".[learning,bench,dev]"` y Jupyter. Las salidas nuevas se guardan en `runs/tesis_figuras/`. No reemplazar resultados históricos sin comprobar el protocolo.

In [ ]:
from pathlib import Path
import sys, subprocess, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'benchmarks'/'common.py').exists() and (p/'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError("Abre Jupyter desde la raíz de point-field o una subcarpeta del repositorio.")
sys.path.insert(0, str(ROOT/'benchmarks'))
OUT = ROOT/'runs'/'tesis_figuras'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'figure.dpi': 120, 'savefig.dpi': 300, 'pdf.fonttype': 42})
COLORS = ['#21618c','#c65036','#63933e','#8b65a5','#c49a38','#458e91']
def save(fig, name):
    for ext in ('pdf','png'):
        fig.savefig(OUT/f'{name}.{ext}', bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
def points(ax, X, y=None):
    if y is None: ax.scatter(*X.T, s=5, c=COLORS[0], linewidths=0)
    else:
        for j,c in enumerate(np.unique(y)):
            ax.scatter(*X[y==c].T, s=5, c='#b9b9b9' if c==-1 else COLORS[j%len(COLORS)], linewidths=0)
    ax.set_xticks([]); ax.set_yticks([])
    if X.shape[1]==2: ax.set_aspect('equal', adjustable='box')
    else: ax.set_zticks([]); ax.set_box_aspect((1,1,1)); ax.view_init(elev=23, azim=35)
def run_script(name, *args):
    subprocess.run([sys.executable, str(ROOT/'benchmarks'/name), *map(str,args)], cwd=ROOT, check=True)
print('Repositorio:', ROOT, '\nSalidas:', OUT)
try:
    commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
except (OSError, subprocess.CalledProcessError):
    commit='No disponible'
print('Commit:',commit)

**1. Construcción visual del descriptor.** Los bloques se extraen mediante la API real. No confundir los valores de los bloques con una reconstrucción de la forma.

In [ ]:
from pointfield.objects import make_object_dataset
from pointfield.learning import CDDescriptor,moment_descriptor,BlockScaler
clouds,y,families=make_object_dataset(dim=3,n_per_class=10,n_points=512,seed=0)
cd=CDDescriptor();F=np.array([cd(x) for x in clouds]); M=np.array([moment_descriptor(x) for x in clouds]);blocks=cd.block_slices(3)
print('CD:',F.shape,'Momentos:',M.shape,'Bloques:',blocks)
fig=plt.figure(figsize=(11,5.5),constrained_layout=True);gs=fig.add_gridspec(3,1+len(blocks),width_ratios=[1.3]+[1]*len(blocks))
for i,c in enumerate(np.unique(y)[:3]):
    k=np.flatnonzero(y==c)[0];ax=fig.add_subplot(gs[i,0],projection='3d');points(ax,clouds[k]);ax.set_title(str(families[int(c)]))
    for j,(b,sl) in enumerate(blocks.items(),1):
        ax=fig.add_subplot(gs[i,j]);ax.plot(F[k,sl],'.-',ms=3);ax.set_title(b if i==0 else '');ax.set_xlabel('Índice');ax.grid(alpha=.2)
save(fig,'05_bloques_descriptor')

**2. Visualizar embeddings sintéticos y vecinos.** PCA es una proyección ilustrativa, no la distancia original. Se usa la distancia entre descriptores escalados; aquí se trata de un ejemplo exploratorio sobre una colección común.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
S=BlockScaler(blocks).fit_transform(F); Z=PCA(n_components=2).fit_transform(S)
fig,ax=plt.subplots(figsize=(7,5),constrained_layout=True)
for j,c in enumerate(np.unique(y)):
    ax.scatter(*Z[y==c].T,s=18,label=str(families[int(c)]),c=COLORS[j%len(COLORS)])
ax.set(xlabel='PC1',ylabel='PC2');ax.legend(fontsize=8,frameon=False);save(fig,'05_embedding_sintetico')
D=pairwise_distances(S);np.fill_diagonal(D,np.inf)
fig=plt.figure(figsize=(11,6),constrained_layout=True)
for i,c in enumerate(np.unique(y)[:3]):
    k=np.flatnonzero(y==c)[0]; near=np.argsort(D[k])[:4]
    for j,idx in enumerate([k,*near]):
        ax=fig.add_subplot(3,5,i*5+j+1,projection='3d');points(ax,clouds[idx]);ax.set_title(('Consulta: ' if j==0 else f'{j}: ')+str(families[int(y[idx])]),fontsize=9)
save(fig,'05_vecinos_sinteticos')

**3. Resultados archivados.** Leer las columnas reales de los CSV. Las barras muestran mAP, no exactitud. Los valores históricos pueden pertenecer a protocolos diferentes.

In [ ]:
for file in ['objetos_3d.csv','modelnet10_test.csv','modelnet40_test.csv']:
    path=ROOT/'benchmarks/results'/file
    if not path.exists():print('No disponible:',path);continue
    data=pd.read_csv(path); assert {'condition','method','mAP'}<=set(data.columns)
    metrics=data[data['nn'].notna()].copy() if 'nn' in data.columns else data[~data['method'].str.contains(' - ',regex=False)].copy()
    contrasts=data.loc[~data.index.isin(metrics.index)].copy()
    if not contrasts.empty: contrasts.to_csv(OUT/f'05_{path.stem}_contrastes.csv',index=False)
    table=metrics.pivot(index='method',columns='condition',values='mAP');display(table.round(3));table.to_csv(OUT/f'05_{path.stem}_tabla.csv')
    fig,ax=plt.subplots(figsize=(10,5),constrained_layout=True);table.plot.bar(ax=ax,width=.85);ax.set_ylabel('mAP');ax.set_ylim(0,1);ax.legend(title='Condición',frameon=False,ncol=3);ax.tick_params(axis='x',rotation=30);save(fig,'05_'+path.stem)

**4. Reejecución opcional.** Los scripts públicos admiten `--out`. ModelNet requiere mallas OFF locales; para ModelNet40 se reutiliza el lector genérico, con tamaños explícitos. No es una réplica certificada del protocolo anterior.

In [ ]:
RUN_SYNTHETIC=False
RUN_MODELNET10=False
RUN_MODELNET40=False
if RUN_SYNTHETIC:run_script('objetos_invariantes.py','--dim',3,'--out',OUT/'05_objetos_3d_nuevo.csv')
for enabled,folder,nref,ntest in [(RUN_MODELNET10,'ModelNet10',400,200),(RUN_MODELNET40,'ModelNet40',1000,400)]:
    if enabled:
        root=ROOT/'data'/folder
        if not root.is_dir():raise FileNotFoundError(root)
        run_script('modelnet10.py','--root',root,'--split','test','--n-ref',nref,'--n-test',ntest,'--out',OUT/f'05_{folder}_nuevo.csv')
print('Activa las variables RUN correspondientes para recalcular; no se sobrescriben CSV históricos.')